<a href="https://colab.research.google.com/github/DivyaAnkam/AI_DecisionTree/blob/main/Divya_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Hugging Face Login (Required for Gated Models like Llama-3.1)

In [15]:
from huggingface_hub import login
from google.colab import userdata

# Retrieve the Hugging Face token from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Log in to Hugging Face
if hf_token:
    login(token=hf_token)
    print("Successfully logged in to Hugging Face!")
else:
    print("Hugging Face token not found in Colab secrets. Please add it as 'HF_TOKEN'.")

SecretNotFoundError: Secret HF_TOKEN does not exist.

In [5]:
#print('NOTE: Intentionally crashing session to use the newly installed library.\n')

!pip uninstall -y pyarrow
!pip install ray[debug]==2.31.0
!pip install bs4

# A hack to force the runtime to restart, needed to include the above dependencies.
#import os
#os._exit(0)
import os
import ray


Found existing installation: pyarrow 23.0.1
Uninstalling pyarrow-23.0.1:
  Successfully uninstalled pyarrow-23.0.1


In [ ]:
#print('Uninstalling existing torch installations...')
#!pip uninstall torch torchvision torchaudio -y

print('Installing the latest stable PyTorch...')
!pip install torch --no-cache-dir

#print('\n*** IMPORTANT: Please restart your Colab runtime now (Runtime > Restart runtime) to complete the PyTorch installation and clear any lingering conflicts. ***')

In [6]:
# Force uninstall everything torch-related
!pip uninstall torch torchvision torchaudio -y

# Sometimes a second run finds "shadow" versions
!pip uninstall torch -y

# Reinstall a clean, stable version
!pip install torch --no-cache-dir


Found existing installation: torch 2.11.0
Uninstalling torch-2.11.0:
  Successfully uninstalled torch-2.11.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 72.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.26 requires torchvision, which is not installed.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.


In [2]:
!pip install "ray[train]"

  Using cached pyarrow-23.0.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.1 kB)
Using cached pyarrow-23.0.1-cp312-cp312-manylinux_2_28_x86_64.whl (47.6 MB)


In [3]:
#!pip install torch --no-cache-dir
import os
import ray
import torch
from ray import train
from ray.train.torch import TorchTrainer
from ray.train.torch import TorchConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel, get_peft_model

In [ ]:
print('NOTE: Intentionally crashing session to use the newly installed library.\n')

!pip uninstall -y pyarrow
!pip install ray[debug]==2.31.0
!pip install bs4

# A hack to force the runtime to restart, needed to include the above dependencies.
#import os
#os._exit(0)

In [5]:
# Initialize Ray
num_gpus = torch.cuda.device_count()
ray.init(
    object_store_memory=2 * 1024 * 1024 * 1024,
    num_cpus=20,
    num_gpus=num_gpus
)
# Configuration for federated learning
fed_config = {
    "base_model": "meta-llama/Llama-3.1-8B-Instruct",
    "num_epochs": 0.5,
    "batch_size": 4,
    "learning_rate": 2e-5,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"]
}

2026-04-07 19:39:57,654	INFO worker.py:1771 -- Started a local Ray instance.


In [6]:
def train_on_local_documents(config):
    # Get worker-specific device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Set up local model with LoRA
    base_model = AutoModelForCausalLM.from_pretrained(
        config["base_model"],
        device_map="auto",
        load_in_4bit=True
    )

    # Configure LoRA adapter
    lora_config = LoraConfig(
        r=config["lora_r"],
        lora_alpha=config["lora_alpha"],
        target_modules=config["target_modules"],
        lora_dropout=config["lora_dropout"],
        bias="none",
        task_type="CAUSAL_LM"
    )

    # Apply LoRA to model
    model = get_peft_model(base_model, lora_config)
    model = model.to(device)

    # Set up tokenizer
    tokenizer = AutoTokenizer.from_pretrained(config["base_model"])

    # Get worker ID to identify which partition to use
    worker_id = train.get_context().get_worker_id()

    # Process local documents (adapted from existing pipeline)
    local_data = process_local_documents(f"Documents_partition_{worker_id}")

    # Convert to dataset
    dataset = convert_to_dataset(local_data)

    # Training arguments
    training_args = TrainingArguments(
        output_dir=f"./worker_{worker_id}_output",
        per_device_train_batch_size=config["batch_size"],
        learning_rate=config["learning_rate"],
        num_train_epochs=config["num_epochs"],
        gradient_accumulation_steps=8,
        optim="adamw_torch",
        report_to="none",
    )

    # Initialize trainer
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        args=training_args,
    )

    # Train model
    trainer.train()

    # Extract LoRA adapter weights only (much smaller than full model)
    adapter_weights = extract_lora_weights(model)

    # Report weights back to aggregator
    train.report({"adapter_weights": adapter_weights})

In [7]:
def partition_documents(source_dir, num_partitions):
    """Partition documents into separate directories for federated learning"""

    all_files = []
    for root, _, files in os.walk(source_dir):
        for file in files:
            file_path = os.path.join(root, file)
            all_files.append(file_path)

    # Create partition directories
    for i in range(num_partitions):
        os.makedirs(f"Documents_partition_{i}", exist_ok=True)

    # Distribute files across partitions
    for i, file_path in enumerate(all_files):
        partition_idx = i % num_partitions
        dest_path = os.path.join(f"Documents_partition_{partition_idx}",
                                os.path.basename(file_path))
        shutil.copy(file_path, dest_path)

    print(f"Partitioned {len(all_files)} documents into {num_partitions} partitions")


In [8]:
def fedavg_aggregate(weights_list):
    """Implement Federated Averaging (FedAvg) algorithm"""

    # Initialize aggregated weights with the first model's weights
    aggregated_weights = {k: torch.zeros_like(v) for k, v in weights_list[0].items()}

    # Sum all weights
    for weights in weights_list:
        for key in aggregated_weights:
            aggregated_weights[key] += weights[key]

    # Average the weights
    num_models = len(weights_list)
    for key in aggregated_weights:
        aggregated_weights[key] /= num_models

    return aggregated_weights

In [9]:
def apply_aggregated_weights(base_model, lora_config, aggregated_weights):
    """Apply aggregated weights to a fresh model"""

    model = get_peft_model(base_model, lora_config)

    # Load the aggregated weights
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in aggregated_weights:
                param.copy_(aggregated_weights[name])

    return model

In [10]:
def run_federated_training():
    # 1. Partition the data
    # Use 1 partition if num_gpus is 0, otherwise use num_gpus
    num_partitions_for_data = num_gpus if num_gpus > 0 else 1
    partition_documents("Documents_semi_structured", num_partitions_for_data)

    # Determine num_workers and use_gpu based on actual num_gpus
    actual_num_workers = num_gpus if num_gpus > 0 else 1 # At least 1 worker for CPU training
    actual_use_gpu = True if num_gpus > 0 else False
    backend_config = "nccl" if num_gpus > 0 else "gloo" # Use gloo backend for CPU

    # 2. Setup the trainer
    trainer = TorchTrainer(
        train_on_local_documents,
        train_loop_config=fed_config,
        scaling_config=train.ScalingConfig(
            num_workers=actual_num_workers,
            use_gpu=actual_use_gpu,
        ),
        torch_config=TorchConfig(backend=backend_config)
    )

    # 3. Run federated training
    results = trainer.fit()

    # 4. Extract all worker model weights
    worker_weights = [result["adapter_weights"] for result in results.metrics_dataframe.to_dict('records')]

    # 5. Aggregate weights using FedAvg
    aggregated_weights = fedavg_aggregate(worker_weights)

    # 6. Create and save the final model
    base_model = AutoModelForCausalLM.from_pretrained(
        fed_config["base_model"],
        device_map="auto",
        load_in_4bit=True
    )

    lora_config = LoraConfig(
        r=fed_config["lora_r"],
        lora_alpha=fed_config["lora_alpha"],
        target_modules=fed_config["target_modules"],
        lora_dropout=fed_config["lora_dropout"],
        bias="none",
        task_type="CAUSAL_LM"
    )

    final_model = apply_aggregated_weights(base_model, lora_config, aggregated_weights)

    # 7. Save the final model
    save_path = "./federated_model"
    final_model.save_pretrained(save_path)

    return save_path

In [11]:
def federated_evaluation(model_path):
    """Run distributed evaluation on the federated model"""

    @ray.remote(num_gpus=1)
    def evaluate_on_partition(partition_id, model_path):
        # Load model
        model = PeftModel.from_pretrained(
            AutoModelForCausalLM.from_pretrained(
                fed_config["base_model"],
                device_map="auto",
                load_in_4bit=True
            ),
            model_path
        )

        # Load eval datasets for this partition
        eval_datasets = load_evaluation_datasets(f"eval_partition_{partition_id}")

        # Run evaluation
        metrics = {}
        for dataset_name, dataset in eval_datasets.items():
            # Implement evaluation logic similar to existing code
            dataset_metrics = evaluate_dataset(model, dataset)
            metrics[dataset_name] = dataset_metrics

        return metrics

    # Distribute evaluation across GPUs
    futures = [evaluate_on_partition.remote(i, model_path) for i in range(num_gpus)]
    results = ray.get(futures)

    # Combine results
    combined_metrics = {}
    for result in results:
        for dataset_name, metrics in result.items():
            if dataset_name not in combined_metrics:
                combined_metrics[dataset_name] = []
            combined_metrics[dataset_name].append(metrics)

    # Average metrics across partitions
    final_metrics = {}
    for dataset_name, metrics_list in combined_metrics.items():
        final_metrics[dataset_name] = {
            metric: sum(m[metric] for m in metrics_list) / len(metrics_list)
            for metric in metrics_list[0]
        }

    return final_metrics

In [ ]:
# Install additional necessary libraries
!pip install trl datasets accelerate bitsandbytes

In [12]:
import os
import shutil
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Placeholder for process_local_documents
def process_local_documents(directory_path):
    print(f"Processing documents in {directory_path}")
    texts = []
    # Simulate reading some data
    if not os.path.exists(directory_path):
        print(f"Directory {directory_path} not found. Creating dummy data.")
        os.makedirs(directory_path, exist_ok=True)
        with open(os.path.join(directory_path, "doc1.txt"), "w") as f:
            f.write("This is a sample document for federated learning training. It contains some text.")
        with open(os.path.join(directory_path, "doc2.txt"), "w") as f:
            f.write("Another example document to demonstrate the process. More text here.")

    for filename in os.listdir(directory_path):
        if filename.endswith(".txt"):
            with open(os.path.join(directory_path, filename), "r") as f:
                texts.append(f.read())

    # Dummy tokenization/formatting for training (SFTTrainer expects text usually)
    formatted_texts = [{"text": t} for t in texts]
    return formatted_texts

# Placeholder for convert_to_dataset
def convert_to_dataset(data):
    print("Converting data to dataset")
    # Ensure data is in the expected format for Dataset.from_list
    return Dataset.from_list(data)

# Placeholder for extract_lora_weights
def extract_lora_weights(model):
    print("Extracting LoRA weights")
    # This is a simplified extraction. In a real scenario, you'd iterate
    # through `model.named_parameters()` and pick only LoRA-related layers.
    # For demonstration, we'll just return a dummy dict or the full state_dict if needed.
    lora_weights = {name: param.clone().detach().cpu() for name, param in model.named_parameters() if 'lora' in name.lower()}
    if not lora_weights:
        # Fallback if no specific 'lora' layers are found, return dummy
        print("No 'lora' layers found. Returning a dummy weight for demonstration.")
        # Create a dummy weight for aggregation if no lora layers are identified yet
        return {"dummy_lora_param": torch.zeros(1)}
    return lora_weights

# Placeholder for load_evaluation_datasets
def load_evaluation_datasets(partition_name):
    print(f"Loading evaluation datasets for {partition_name}")
    # Create a dummy directory and file if it doesn't exist
    if not os.path.exists(partition_name):
        os.makedirs(partition_name, exist_ok=True)
        with open(os.path.join(partition_name, "eval_doc.txt"), "w") as f:
            f.write("This is an evaluation document. It has some text for testing.")

    # Simulate loading a dataset. In a real scenario, this would be a properly formatted Dataset object.
    dummy_eval_data = [{
        "text": "This is an evaluation example. The model should predict this text."
    }]
    return {"eval_set": Dataset.from_list(dummy_eval_data)}

# Placeholder for evaluate_dataset
def evaluate_dataset(model, dataset):
    print("Evaluating dataset (dummy evaluation)")
    # Simulate evaluation metrics
    return {"perplexity": 10.0, "accuracy": 0.75}

In [13]:
# 1. Create dummy semi-structured documents directory
if os.path.exists("Documents_semi_structured"):
    shutil.rmtree("Documents_semi_structured") # Clean up previous run
os.makedirs("Documents_semi_structured", exist_ok=True)

for i in range(5):
    with open(f"Documents_semi_structured/doc_{i}.txt", "w") as f:
        f.write(f"This is a sample document {i} for federated learning. It contains some unique content for worker {i%2}.\n")

print("Created dummy 'Documents_semi_structured' directory with sample files.")

# 2. Create dummy evaluation partitions (if num_gpus > 0, otherwise 1 partition for CPU)
num_eval_partitions = num_gpus if num_gpus > 0 else 1
for i in range(num_eval_partitions):
    eval_dir = f"eval_partition_{i}"
    if os.path.exists(eval_dir):
        shutil.rmtree(eval_dir)
    os.makedirs(eval_dir, exist_ok=True)
    with open(os.path.join(eval_dir, f"eval_sample_{i}.txt"), "w") as f:
        f.write(f"Evaluation data for partition {i}. This text is for testing the federated model.")
print(f"Created dummy evaluation partitions for {num_eval_partitions} workers.")

Created dummy 'Documents_semi_structured' directory with sample files.
Created dummy evaluation partitions for 1 workers.


In [14]:
# Run the federated training and evaluation
print("Starting federated training...")
saved_model_path = run_federated_training()
print(f"Federated model saved at: {saved_model_path}")

print("Starting federated evaluation...")
final_metrics = federated_evaluation(saved_model_path)
print("Federated Evaluation Results:")
for dataset_name, metrics in final_metrics.items():
    print(f"Dataset: {dataset_name}")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")

Starting federated training...
Partitioned 5 documents into 1 partitions

View detailed results here: /root/ray_results/TorchTrainer_2026-04-07_19-41-17
To visualize your results with TensorBoard, run: `tensorboard --logdir /tmp/ray/session_2026-04-07_19-39-46_062671_57302/artifacts/2026-04-07_19-41-17/TorchTrainer_2026-04-07_19-41-17/driver_artifacts`

Training started with configuration:
+---------------------------------------------------------+
| Training config                                         |
+---------------------------------------------------------+
| train_loop_config/base_model       ...a-3.1-8B-Instruct |
| train_loop_config/batch_size                          4 |
| train_loop_config/learning_rate                   2e-05 |
| train_loop_config/lora_alpha                         32 |
| train_loop_config/lora_dropout                     0.05 |
| train_loop_config/lora_r                             16 |
| train_loop_config/num_epochs                        0.5 |
| train

(TrainTrainable pid=58828) Trainable.setup took 33.873 seconds. If your trainable is slow to initialize, consider setting reuse_actors=True to reduce actor creation overheads.
(TorchTrainer pid=58828) Started distributed worker processes: 
(TorchTrainer pid=58828) - (ip=172.28.0.12, pid=59136) world_rank=0, local_rank=0, node_rank=0
(RayTrainWorker pid=59136) Setting up process group for: env:// [rank=0, world_size=1]
2026-04-07 19:43:06,113	ERROR tune_controller.py:1331 -- Trial task failed for trial TorchTrainer_bea02_00000
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/ray/air/execution/_internal/event_manager.py", line 110, in resolve_future
    result = ray.get(future)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ray/_private/auto_init_hook.py", line 21, in auto_init_wrapper
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ray/_private/client_mode_hook.p


Training errored after 0 iterations at 2026-04-07 19:43:06. Total running time: 1min 37s
Error file: /tmp/ray/session_2026-04-07_19-39-46_062671_57302/artifacts/2026-04-07_19-41-17/TorchTrainer_2026-04-07_19-41-17/driver_artifacts/TorchTrainer_bea02_00000_0_2026-04-07_19-41-28/error.txt



TrainingFailedError: The Ray Train run failed. Please inspect the previous error messages for a cause. After fixing the issue (assuming that the error is not caused by your own application logic, but rather an error such as OOM), you can restart the run from scratch or continue this run.
To continue this run, you can use: `trainer = TorchTrainer.restore("/root/ray_results/TorchTrainer_2026-04-07_19-41-17")`.
To start a new run that will retry on training failures, set `train.RunConfig(failure_config=train.FailureConfig(max_failures))` in the Trainer's `run_config` with `max_failures > 0`, or `max_failures = -1` for unlimited retries.

huggingface login try and save the keys in colab secrets as a token
